In [ ]:
import socket
import time
import csv
import subprocess
import re
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from config import CSV_FILE_APP


In [ ]:
IP = '0.0.0.0'
PORT = 9999
BUFFER_SIZE = 1024
CSV_FILE = CSV_FILE_APP
INTERFACE = 'wlp1s0'  

In [ ]:

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind((IP, PORT))
print(f"[Server] Listening on {IP}:{PORT}")

last_transit = None
expected_seq = 0
jitter_rfc = 0.0

first_packet = True

with open(CSV_FILE, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow([
        'Time',
        'Latency',
        'Jitter',
        'Packet Loss Count'
    ])

    while True:
        data, addr = sock.recvfrom(BUFFER_SIZE)
        arrival_time = time.time()

        try:
            seq_str, sent_str = data.decode().split(',')
            seq = int(seq_str)
            sent_time = float(sent_str)
        except Exception as e:
            print(f"[Error] Invalid packet: {e}")
            continue

        # --- Latency ---
        latency_ms = (arrival_time - sent_time) * 1000

        # --- RFC Jitter ---
        transit = arrival_time - sent_time
        if last_transit is not None:
            d = abs((transit - last_transit) * 1000)
            jitter_rfc += (d - jitter_rfc) / 16
        else:
            jitter_rfc = 0.0
        last_transit = transit

        # --- Packet Loss Detection ---
        if seq > expected_seq:
            lost_packets = seq - expected_seq
        else:
            lost_packets = 0
        expected_seq = seq + 1

        if first_packet:
            first_packet = False
        else:
            writer.writerow([
                datetime.now().strftime('%H:%M:%S.%f')[:-3],
                round(latency_ms, 3),
                round(jitter_rfc, 3),
                lost_packets,
            ])

        print(f"[Recv] Seq={seq} | Latency={latency_ms:.2f} ms | Jitter={jitter_rfc:.2f} ms | "
              f"Lost={lost_packets}")



In [ ]:
df = pd.read_csv(CSV_FILE)

df['Time'] = pd.to_datetime(df['Time'])
start_time = df['Time'].iloc[0]
df['Elapsed'] = (df['Time'] - start_time).dt.total_seconds()

# Plot 1: Latency
plt.figure(figsize=(12, 6))
plt.plot(df['Elapsed'],df['Latency'], color='green')
plt.title('Latency Over Time')
plt.xlabel('Time')
plt.ylabel('Latency (ms)')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Jitter
plt.figure(figsize=(12, 6))
plt.plot(df['Elapsed'], df['Jitter'], color='blue')
plt.title('Jitter Over Time')
plt.xlabel('Time')
plt.ylabel('Jitter (ms)')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 3: Packet Loss
plt.figure(figsize=(12, 6))
plt.plot(df['Elapsed'], df['Packet Loss Count'], color='red', marker='o')
plt.title('Packet Loss Over Time')
plt.xlabel('Time')
plt.ylabel('Lost Packets')
plt.grid(True)
plt.tight_layout()
plt.show()

